# Native-State Marginal Calibration

Reproduce synthetic, budget-only calibration: source schedules and finite beliefs, canonical projection maps, independent rational truths, prepared simulation batches, and independent statistical readback. This is a separate method from the historical capped calibration and is not validation of human observations or a globally controlled error rate. See the [statistical settings](../../configs/README.md#statistical-settings).

For reference-matched reproduction on Linux x86-64, first create and select the numerical environment below. The pins include the BLAS implementation because projection coefficient bits enter the exact truth. Node.js 22.14.0 is the reference JavaScript runtime and must be installed separately.

```sh
conda env create -f configs/native_reference_environment.yml
conda activate native-calibration-reference
python -m ipykernel install --user --name native-calibration-reference --display-name 'Native calibration reference'
```

Open this notebook with the **Native calibration reference** kernel. The first cell installs the project and remaining dependencies without upgrading NumPy or SciPy. The reference build is not an assertion of bitwise equivalence on other platforms.


In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-c', str(ROOT / 'configs/native_reference_constraints.txt'), '-e', str(ROOT)], check=True)
subprocess.run(['node', '--version'], check=True)  # Install Node.js separately if unavailable.


## Settings

`smoke` runs three representative subcases, one full 250-attempt batch each. `full` runs all 40 subcases, 40 batches each: 400,000 total attempts. Both use the same fixed sample sizes, RNG array shapes and seeds. A smoke result is `PROVISIONAL`, not calibration acceptance. `WORKERS` parallelizes independent subcases; choose it for the available memory/CPUs.

The original reference runtime was Python 3.10.9, NumPy 1.23.5, SciPy 1.10.0 and Node 22.14.0. The package supports newer runtimes; finite projection/linear-algebra rounding can differ. Content and version bindings prevent mixing batches across runtimes. Preserve discrepancies rather than rounding native expectations to zero or changing seeds.


In [ ]:
RUN = False
MODE = 'smoke'  # 'full' opts into 400,000 attempts.
WORKERS = 1
PREPARED = ROOT / 'results/native_calibration/prepared'
OUTPUT = ROOT / f'results/native_calibration/{MODE}'
RUNNER = ROOT / 'scripts/reproduce_native_calibration.py'


def invoke(*arguments):
    command = [sys.executable, str(RUNNER), *map(str, arguments)]
    print(' '.join(command))
    if RUN:
        subprocess.run(command, cwd=ROOT, check=True)


## 1. Reconstruct Inputs

Preparation generates 45 genuine synthetic schedules, finite states at prefixes 0/1/3 and exact exhaustion, working/control lookup maps, candidate truths and independent exact native truths. No private input cache is needed. Preparation requires a new directory; reuse an already completed preparation only with the same numerical source/runtime. Incomplete preparation should be kept for diagnosis and a new directory selected.


In [ ]:
if not PREPARED.exists():
    invoke('prepare', '--prepared', PREPARED)
else:
    print('Existing preparation will be content-checked by run/summarize:', PREPARED)


## 2. Generate Calibration Batches

The Normal-reference, native finite-state and selected/missing-data families retain distinct laws. The producer verifies row/person endpoints and scalar/vector tests at batch IDs 0 and 39. It does not cache simulated endpoints. Completed cells are published only after final input checks. Repeating the same command resumes compatible completed batches exactly once. For finer checkpoints, use `--cell CELL:SUBCASE --batch-id ID`; each ID always represents 250 attempts. Different processes must not write the same cell concurrently.


In [ ]:
invoke('run', '--prepared', PREPARED, '--output', OUTPUT, '--mode', MODE, '--workers', WORKERS)


## 3. Independently Read Back Results

The reader recalculates Clopper-Pearson bounds with independent continued fractions and eligible Normal-reference power by integration. Rejection uses all attempts; coverage uses available intervals. Exact rational truths are compared to the binary64 interval endpoints. Nonzero native expectations remain non-boundary, even if their displayed value is tiny.


In [ ]:
invoke('summarize', '--prepared', PREPARED, '--output', OUTPUT, '--mode', MODE)

if RUN:
    import json
    import pandas as pd
    summary = json.loads((OUTPUT / 'summary.json').read_text())
    print({k: v for k, v in summary.items() if k != 'cells'})
    display(pd.read_csv(OUTPUT / 'targets.csv'))


## Interpretation

The full reference calculation had 118 target rows: 89 `PASS` (including five nonprimary D10 rows), 22 `UNAVAILABLE`, and seven `DIAGNOSTIC_ONLY`. These are scoped calibration screens, not universal validity. The 22 unavailable rows have nonzero exact native means rather than exact boundary-null targets; their available-interval coverage can still pass. C04 does not have a derived coverage truth.

Earlier historical failures remain relevant: 94 selection-diagnostic rows and 17 other unmatched rows are not repaired by this calibration. Selected-population calibration does not restore the original-population estimand. Finite native truth enumerates response patterns under declared ideal probability laws, not the literal finite PRNG distribution. No family-wise error, simultaneous coverage, latest cap20 compatibility or cost-aware adaptive-policy claim follows.

All outputs are generated under ignored `results/`; model/source logic stays in `src/tutoring_analysis`.
